In [2]:
import importlib.metadata
from rich.console import Console
from rich.panel import Panel

from pyrit.prompt_target import OpenAIChatTarget
from pyrit.prompt_normalizer import PromptNormalizer
from pyrit.memory import SQLiteMemory, CentralMemory
from pyrit.models import Message, MessagePiece
from pyrit.score import SelfAskTrueFalseScorer, TrueFalseQuestion
from pyrit.converter import Base64Converter

version = importlib.metadata.version("pyrit")
Console().print(Panel(
    f"[bold green]✓ Environment Ready[/]\n\n"
    f"PyRIT Version  [cyan]{version}[/]\n"
    f"Core Modules   [green]Successfully loaded[/]",
    title="[bold]PyRIT Security Lab[/]",
    border_style="blue"
))

╭────────────────────────────────────────────── PyRIT Security Lab ───────────────────────────────────────────────╮
│ ✓ Environment Ready                                                                                             │
│                                                                                                                 │
│ PyRIT Version  1.1.0                                                                                            │
│ Core Modules   Successfully loaded                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from pathlib import Path

console = Console()
env_path = Path(".env")

if not env_path.exists():
    env_path.write_text("""# PyRIT Course - API Keys
GOOGLE_API_KEY=sk-your-gemini-key-here
GROQ_API_KEY=gsk_your-groq-key-here
""")

    console.print(Panel(
        "[bold green]✓ Environment File Created[/]\n\n"
        "File    [cyan].env[/]\n"
        "Status  [yellow]API keys required[/]\n\n"
        "[dim]Add your Gemini and Groq API keys to continue.[/]",
        title="[bold]Environment Configuration[/]",
        border_style="blue"
    ))
else:
    console.print(Panel(
        "[bold green]✓ Environment File Detected[/]\n\n"
        "File    [cyan].env[/]\n"
        "Status  [green]Ready for configuration[/]",
        title="[bold]Environment Configuration[/]",
        border_style="blue"
    ))

╭─────────────────────────────────────────── Environment Configuration ───────────────────────────────────────────╮
│ ✓ Environment File Detected                                                                                     │
│                                                                                                                 │
│ File    .env                                                                                                    │
│ Status  Ready for configuration                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
import nest_asyncio
from pyrit.memory import SQLiteMemory, CentralMemory

nest_asyncio.apply()

memory = SQLiteMemory()
CentralMemory.set_memory_instance(memory)

existing = memory.get_message_pieces()

Console().print(Panel(
    "[bold green]✓ Memory Initialized[/]\n\n"
    f"Backend   [cyan]{type(memory).__name__}[/]\n"
    f"Storage   [cyan]SQLite[/]\n"
    f"Records   [yellow]{len(existing)}[/]",
    title="[bold]PyRIT Memory Store[/]",
    border_style="blue"
))

[pyrit:alembic] No new upgrade operations detected.


╭────────────────────────────────────────────── PyRIT Memory Store ───────────────────────────────────────────────╮
│ ✓ Memory Initialized                                                                                            │
│                                                                                                                 │
│ Backend   SQLiteMemory                                                                                          │
│ Storage   SQLite                                                                                                │
│ Records   0                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
import os
from dotenv import load_dotenv

from pyrit.prompt_target import LiteLLMChatTarget
from pyrit.prompt_normalizer import PromptNormalizer
from pyrit.models import Message, MessagePiece

# Load environment variables from .env
load_dotenv()

console = Console()

# Gemini target
gemini_target = LiteLLMChatTarget(
    model_name="gemini/gemini-2.5-flash",
    api_key=os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
)

normalizer = PromptNormalizer()

console.print(Panel(
    "[bold green]✓ Target Initialized[/]\n\n"
    "Provider   [cyan]Google Gemini[/]\n"
    "Model      [cyan]Gemini 2.5 Flash[/]\n"
    "Interface  [cyan]LiteLLMChatTarget[/]",
    title="[bold]PyRIT Target Configuration[/]",
    border_style="blue"
))

# Build prompt
msg = Message(message_pieces=[
    MessagePiece(
        role="user",
        original_value="What is artificial intelligence? Answer in exactly 2 sentences."
    )
])

console.print("\n[dim]Sending prompt via PyRIT PromptNormalizer...[/]")

response = await normalizer.send_prompt_async(
    message=msg,
    target=gemini_target
)

# Pretty response
answer = "\n".join(
    piece.original_value
    for piece in response.message_pieces
    if piece.role == "assistant"
)

console.print(Panel(
    answer,
    title="[bold green]Gemini Response[/]",
    border_style="green"
))


╭────────────────────────────────────────── PyRIT Target Configuration ───────────────────────────────────────────╮
│ ✓ Target Initialized                                                                                            │
│                                                                                                                 │
│ Provider   Google Gemini                                                                                        │
│ Model      Gemini 2.5 Flash                                                                                     │
│ Interface  LiteLLMChatTarget                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sending prompt via PyRIT PromptNormalizer...

╭──────────────────────────────────────────────── Gemini Response ────────────────────────────────────────────────╮
│ Artificial intelligence (AI) is the development of computer systems capable of performing tasks that typically  │
│ require human intelligence. These systems can learn, reason, solve problems, perceive, and understand language, │
│ often by recognizing patterns and making predictions from data.                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

console.print(Panel(
    f"[bold green]✓ Response Inspected[/]\n\n"
    f"Type     [cyan]{type(response).__name__}[/]\n"
    f"Pieces   [yellow]{len(response.message_pieces)}[/]",
    title="[bold]Response Object[/]",
    border_style="blue"
))

for i, piece in enumerate(response.message_pieces, 1):
    table = Table(
        title=f"MessagePiece #{i}",
        show_header=True,
        header_style="bold cyan"
    )
    table.add_column("Property", style="dim")
    table.add_column("Value")

    table.add_row("Role", str(piece.role))
    table.add_row("Original Value", str(piece.original_value)[:120])
    table.add_row("Data Type", str(piece.original_value_data_type))
    table.add_row("Conversation ID", str(piece.conversation_id))
    table.add_row("Timestamp", str(piece.timestamp))

    console.print(table)

╭──────────────────────────────────────────────── Response Object ────────────────────────────────────────────────╮
│ ✓ Response Inspected                                                                                            │
│                                                                                                                 │
│ Type     Message                                                                                                │
│ Pieces   1                                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                  MessagePiece #1                                                  
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property        ┃ Value                                                                                         ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Role            │ assistant                                                                                     │
│ Original Value  │ Artificial intelligence (AI) is the development of computer systems capable of performing     │
│                 │ tasks that typically require h                                                                │
│ Data Type       │ text                                                                                          │
│ Conversation ID │ 12a14c46-c665-4ae6-8a8d-336e23ada3e3                                                          │
│ Timestamp       │ 2026-09-29 06:51:41.561316+00:00                                                              │
└─────────────────┴───────────────────────────────────────────────────────────────────────────────────────────────┘

In [9]:
import os
from rich.console import Console
from rich.panel import Panel
from pyrit.prompt_target import OpenAIChatTarget


console = Console()

# Groq is OpenAI-compatible, so OpenAIChatTarget can be used
groq_target = OpenAIChatTarget(
    model_name="qwen/qwen3.8-27b",
    endpoint="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

groq_normalizer = PromptNormalizer()

console.print(Panel(
    "[bold green]✓ Target Initialized[/]\n\n"
    "Provider   [cyan]Groq[/]\n"
    "Model      [cyan]Qwen 3.8 27B[/]\n"
    "Interface  [cyan]OpenAIChatTarget[/]",
    title="[bold]PyRIT Target Configuration[/]",
    border_style="blue"
))

groq_msg = Message(message_pieces=[
    MessagePiece(
        role="user",
        original_value="What is artificial intelligence? Answer in exactly 2 sentences."
    )
])

console.print("\n[dim]Sending prompt via PyRIT PromptNormalizer...[/]")

groq_response = await groq_normalizer.send_prompt_async(
    message=groq_msg,
    target=groq_target
)

answer = "\n".join(
    piece.original_value
    for piece in groq_response.message_pieces
    if piece.role == "assistant"
)

console.print(Panel(
    answer,
    title="[bold green]Qwen Response[/]",
    border_style="green"
))

╭────────────────────────────────────────── PyRIT Target Configuration ───────────────────────────────────────────╮
│ ✓ Target Initialized                                                                                            │
│                                                                                                                 │
│ Provider   Groq                                                                                                 │
│ Model      Qwen 3.8 27B                                                                                         │
│ Interface  OpenAIChatTarget                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sending prompt via PyRIT PromptNormalizer...

╭───────────────────────────────────────────────── Qwen Response ─────────────────────────────────────────────────╮
│ Artificial intelligence is the simulation of human intelligence processes by computer systems, enabling them to │
│ learn, reason, solve problems, and make decisions. This technology encompasses a broad range of applications,   │
│ from natural language processing and computer vision to autonomous robotics.                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from pyrit.memory import CentralMemory

console = Console()

# Retrieve memory and stored messages
memory = CentralMemory.get_memory_instance()
all_pieces = memory.get_message_pieces()

# Memory statistics
conv_ids = {p.conversation_id for p in all_pieces}
user_msgs = [p for p in all_pieces if p.role == "user"]
assistant_msgs = [p for p in all_pieces if p.role == "assistant"]

console.print(Panel(
    "[bold green]✓ Memory Store Inspected[/]\n\n"
    f"Records        [cyan]{len(all_pieces)}[/]\n"
    f"Conversations  [cyan]{len(conv_ids)}[/]\n"
    f"User Messages  [yellow]{len(user_msgs)}[/]\n"
    f"AI Messages    [yellow]{len(assistant_msgs)}[/]",
    title="[bold]SQLite Memory Overview[/]",
    border_style="blue"
))

# Recent messages
table = Table(
    title="Recent Message Pieces",
    header_style="bold cyan"
)
table.add_column("Role", width=12)
table.add_column("Message")
table.add_column("Conversation ID", style="dim")

for piece in all_pieces[-4:]:
    role = "USER" if piece.role == "user" else "ASSISTANT"
    text = str(piece.original_value)[:80].replace("\n", " ")
    conv_id = f"{piece.conversation_id[:16]}..."

    table.add_row(role, text, conv_id)

console.print(table)

╭──────────────────────────────────────────── SQLite Memory Overview ─────────────────────────────────────────────╮
│ ✓ Memory Store Inspected                                                                                        │
│                                                                                                                 │
│ Records        6                                                                                                │
│ Conversations  3                                                                                                │
│ User Messages  3                                                                                                │
│ AI Messages    3                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                               Recent Message Pieces                                               
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Role         ┃ Message                                                                    ┃ Conversation ID     ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ USER         │ What is artificial intelligence? Answer in exactly 2 sentences.            │ 12a14c46-c665-4a... │
│ ASSISTANT    │ Artificial intelligence (AI) is the development of computer systems        │ 12a14c46-c665-4a... │
│              │ capable of p                                                               │                     │
│ USER         │ What is artificial intelligence? Answer in exactly 2 sentences.            │ 84385db8-ad3f-45... │
│ ASSISTANT    │ Artificial intelligence is the simulation of human intelligence processes  │ 84385db8-ad3f-45... │
│              │ by com                                                                     │                     │
└──────────────┴────────────────────────────────────────────────────────────────────────────┴─────────────────────┘